# **Notebook**

## Week 1

**Vraag:** Kunnen we een simulatie maken waarin cellen ontstaan, profileren, communiceren en reageren indien een tumor aanwezig is.

We kunnen de algemene mutatie rate nemen om te zien wanneer een mutatie zou kunnen onstaan en aan de hand hiervan bepalen we dat deze mutatie eigenlijk altijd een tumor mutatie zal zijn.


De cell zal 3 mogelijk statussen hebben.
0: geen cell aanwezig
1: gezonde cell
2: tumor cell

De reden hiervoor is zodat de cell communicatie makkelijk gemoduleerd kan worden. Cellen communiceren met elkaar door signalling pathways. Deze bestaan uit chemische signalen zoals hormonen en eiwitten die door receptoren opgevangen worden. De cellen hebben verschillende soorten contact.

Direct contact is door cellen die strak tegen elkaar aanliggen en communiceren door kanalen die door de celwand heen gaan.

We kunnen door middel van een GUI met bijvoorbeeld tkinter een enorm grid maken gebaseerd op onze gewenste cell grote.

Ik neem aan dat we een "biopt" hebben met een grootte van 50 x 50 pixels, hiervan kunnen we een grid maken waarin we elke column en row kunnen aanroepen. Hiermee kunnen we elke grid 1 visuele pixel aanwijzen en een kleur geven. Eerder benoemd hebben we elk cell type een nummer gegeven. Dit betekent dat we ook elk nummer een kleur kunnen geven.

Er zijn een paar belangerijke dingen om vast te stellen. Hoe modeleer je de cell omgeving? Hoe prolifeerd een tumor cell, welke soort kanker is het meest realistisch om te gebruiken? Hoe willen we dat cellen met elkaar communiceren en nog wat andere vragen en problemen die tijdens het process duidelijk zullen worden.

De NCI - National Cancer Institute (_Common cancer types_.) - zegt de volgende top 5 voor de meest voorkomende kankers: borst- en prostaatkanker op nummer 1, gevolgd door long- en colontumor. Het zal dus 1 van deze 4 worden.

---

We kiezen de Agent-Based Modelling-approach en niet de ODE-weg, maar we houden toch wat inspiratie van de ODE om de regels te schrijven voor de ABM. We nemen als inspiratie het Gompertz-model aan. Dit model zegt dat tumorcellen langzaam beginnen te groeien, waarna er een stijging is, des te meer cellen aanwezig zijn, en deze stijging sluit ook weer af, waardoor een soort S-curve ontstaat. (Yin et al., 2019) (Srinivasa et al., 2020).

We moeten denken aan de staten van de cellen, zoals eerder genoemd kunnen de cellen 3 staten hebben. 0 (leeg), 1 (gezond), 2 (tumor).

Een lege stuurt geen signalen, maar kan wel een signaal ontvangen om te veranderen naar een gezonde cell of tumorcel.

Een gezonde cell kan signalen sturen naar lege cellen, en signalen ontvangen van tumor cellen om hun staat te veranderen

Een tumorcel kan signalen versturen naar gezonde cellen en lege cellen om ze te veranderen of te laten groeien tot een tumorcel.

(Misschien laten nog chemo inplannen die tumorcellen dood of veranderd naar een gezonde cell.)

---

De grid heeft een klein probleem. Laten we zeggen dat we een grid hebben van 50 bij 50 en er is op positie 24 een tumorcel. Dit kan zeggen dat de cel 25 hierdoor ook een tumorcel is. Waardoor als we cel 25 lezen ook als een tumorcel de 26 cell die gezond is ook zal veranderen waardoor we onrealistisch worden. Hierdoor hebben we 2 grids nodig. De oude en de nieuwe.

Dit kunnen we doen als volgt:
```python
def create_grid(rows, cols):
    grid = [[0 for _ in range(cols)] for _ in range(rows)]
    return grid

def copy_grid(grid):
    copied_grid = [row[:] for row in grid]
    return copied_grid

```

---

Maken we een "thin-agent" of een "fat-agent"



We kunnen nu door middel van de aangegeven grootte van de grid een tumor plaatsen in de grid.
```python
def create_grid(rows, cols):
    grid = [[0 for _ in range(cols)] for _ in range(rows)]
    return grid

def copy_grid(grid):
    copied_grid = [row[:] for row in grid]
    return copied_grid

def place_tumor(grid, rows, cols):
    row = random.randint(0, rows - 1)
    col = random.randint(0, cols - 1)
    grid[row][col] = 2
    return grid
```


---
We moeten kijken hoe we de tijdstappen maken voor de simulatie

De timeloop moet elke cell checken voor welke status hij bevat 0, 1 of 2 en aan de hand daar van moet de cell communicatie aangeroepen worden.

Als we kijken naar het environment file hebben we de algemene logica voor de verspreiding van de tumor gemaakt.

De tumor zal kijken naar de cellen naast zich om te checken of ze gezond zijn, al een tumor zijn of dat er geen cellen aanwezig zijn.

De tumor zal verspreiden als de cell een status van 1 of 0 heeft, maar zal de cell overslaan indien de cell all een tumor cell is.

Het mechanisme is op dit moment een while loop met een nested for loop wat betekent dat de big O hoog is. Later zal optimalisatie nodig hebben echter is het werkend krijgen opdit moment de prioriteit.

---

Tumors zullen niet gegarandeerd elke tijdstap verspreiden als we ons Gompertz model aanhouden. Dit betekent dat we een P_INFECTION zullen opzetten wat de infectie probability (kans) zal opzetten.

# Week 2

Terug komende op hoe de tumor verspreid. Een tumor "duwt" andere cellen uit de weg (Jain et al., 2014). In mijn model lijkt het nu of de cellen een soort "geinfecteerd" worden waardoor ze naar een tumor cell veranderen. Het moet dus eigenlijk aangepast worden om te laten zien dat de cellen een soort weggeduwt worden door de tumor?

We kunnen zeggen dat in onze matrix de tumor de cell zal "overnemen" en dat dit de "Duw" is zoals beschreven in (Jain et al., 2014). Dit moet wel duidelijk vermeld worden in de readme.

Nu dat we weten dat de tumor de andere cellen uit de weg duwt moeten we wel kijken hoe vaak dit gebeurt per tijd stap. Een soort tumor groei kans per tijdstap.

Het is tijd om een type kanker uit te zoeken omdat we graag de formule of wel groei snelheid willen vaststellen. Aan de hand van het eerdere onderzoek hebben vast gesteld dat long kanker een type kanker is wat vaak voorkomt. Ik ga in de long kanker voor non-small lung cancer omdat dit binnen de long kankers het meest voorkomende type is. (Lung Cancer Foundation of America, 2024)

We willen nu kijken wat de doubling time, is van de non small lung cancer hiermee kunnen we berekenen hoe we onze "verspreiding" van onze tumor gaan modelleren.

Volgens Jiang et al., 2024, is de verdubbeling tijd van onder andere non small lung cancer median 207 dagen is. Zoals we weten uit de gompertz is dat de tumor initieel heel sloom zal groeien, maar meer tumor cellen betekent meer verspreiding. De formule die we gaan gebruiken is als volgt

(1 - P)^207 = 0.5 ook wel P = 1 - (0.5 ** (1/207)) in python dit geeft ons het volgende getal: 0.003342937006.
Dit getal betekent biologisch de kans dat de tumor zich zal delen per dag is 0.33%.

We modelleren celdeling en expansie als 1 gecombineerd proces waarbij een tumorcel direct een buurcel overneemt. Het fysiek verplaatsen van cellen zou computationeel veel duurder zijn zonder de biologische accuraatheid te verbeteren. Dit is dus een bewuste modelaanname.

In deze simulatie willen we ook dat de gezonde cellen groeien natuurlijk, omdat deze simulatie ook een normale proliferatie heeft. Dit kunnen we doen op dezelefde manier door te kijken naar de dubbeling tijd van long weefsel en met dezelfde berekening als bij de tumor verdubbeling tijd te doen. Het is long weefsel omdat we eerder vast gesteld hebben dat het een long tumor is.

Normale gezond long weefsel in volwassenen groeien/verdubbelen niet, dus inplaats hier van nemen we aan voor deze simulatie dat de long cellen slow dividing cells zijn zoals lever cellen. We weten dat lever cellen zichzelf regeneren en dat longcellen dit niet doen, maar om de simulatie interesanter te maken, maken we deze verandering voor de modelkeuze.

Levers zijn voornamelijk opgemaakt uit hepatocyten ongeveer 80% (Kalra et al., 2023), Deze cellen verdubbelen ongeveer elke 30-49 uur, maar laten we voor onze simulatie elke twee dagen aan nemen. (Levy et al., 2015)

Dit geeft de volgende formule

(1 - P)^2 = 0.5 dus P = 1 - (0.5 ** (1/2)) = 0.2928932188 dit geeft ons de C_DOUBLING_TIME = 0.2928932188

# Week 3

We hebben nu de algemene uitlijn en werking van het programma, voordat we dit verder maken gaan we nu aan de slag met het displayen van onze grid.

Dit is te doen doormiddel van pygame. Dit kan mooi de grid laten zien en de pixels met de juiste kleuren vervangen.


De time_loop, deze loop berekent in 1 keer de gehele tumor ontwikkeling voor x dagen. Echter, omdat we pygame gebruiken wordt er nog een loop gemaakt waardoor het niet goed zal werken. Hierom passen we dit aan naar for oops waardoor we de while loop van de gameloop gaan gebruiken.


De cell.py class. Initeel was de ABM-gedachte dat elke cell een uniek brein heeft. Dus elke cell in de grid had de cell class aangeroepen. Echter zijn de volgende dingen nodig voor een agent based model. Een agent is een autonomie entiteit die in een status zich bevindt of wel 0, 1 of 2. De agent moet in een omgeving zitten ook wel het environment.py en regels heeft bij ons de get_neighbours.

Voor efficient zal de cell.py dus weggehaald worden.

Nu de gameloop, in de gui.py hebben we een gameloop gemaakt dit is de eerder benoemde loop die gebruikt wordt om de gehele simulatie te runnen. Deze loopt stopt pas bij gebruiker onderbreking door op de sluiting button te drukken wat betekent dat de simulatie start bij het starten van de programma en stops wanneer het programma gesloten wordt.

Voor de fine tuning moeten er nog een paar dingen toegevoegd worden. Een soort legenda met statistiek zou handig zijn, we willen namelijk zien hoeveel tumors er aanwezig zijn, wat de tijdstap is (dag of week) en nog wat andere info. Misschien een pauze button? Zodat we de tumor spreiding rustig kunnen analyseren. Een screengrab? De tumor spreiding is nu een vast cijfer maar in de toekomst zou die naar een input based variabele vervangen kunnen worden zodat meerdere tumors kunnen geanalyseerd worden.

We hebben nu de gompertz formule $N(t)=N_{max.}⋅e^{−b⋅e ^{−c⋅t}}$, de reden om dit type Gompertz te gebruiken is omdat dit een standaard model is voor tumor groei. Deze interpretatie is gemakkelijk te coderen.

We hebben hiervoor een grafiek functie gemaakt die leidt naar een pdf-maker waarin de groei van onze simulatie naast het gompertz model leggen om deze twee vergelijkbaar te maken.

Na wat nadenken is de logica van cell regeneratie niet logisch als we de long kanker hier voor gebruiken. Dus vanaf nu zal de standaard tumor groei snelheid komen van Hepatocellular carcinoma in plaats van non small cell long kanker. Naast deze verandering is het ook mogelijk voor de gebruiker zelf om de groei kans van de tumor aan te passen, maar de eerder benoemde lever kanker zal dus als default gebruikt worden.

De volgende papers (Nathani et al., 2020)(Wigg et al., 2010) (Nathani et al., 2021) zeggen een doubling time van 130 dagen tot 150 dagen laten we dus uitgaan van 140 dagen.

$(1 - P)^140 = 0.5$ ook wel $P = 1 - (0.5 ** (1/140))$





Ik heb command line toegevoegd dit is omdat we de gebruiker de kans willen geven om alles aan te passen zodat er verschillende kanker soorten gesimuleerd kunnen worden.


Eerder in dit document is besproken dat we aannemen dat de simulatie in een "biopt" gebeurt. Dit betekent dat we ook dit ook terug moeten laten zien in het model. Dit kan door "noise" toe tevoegen aan de rand van de grid in de vorm van dode cellen. Dit wordt in de create_grid() gedaan na het aanmaken van het gezonde grid - de cellen met status 1 worden er een aantal 1 status cellen aan de rand van de grid 0 - 8 random overschreven tot 0 status cellen.

Er moet nog een safety aagemaakt worden voor wanneer grids te klein zijn hiervoor. Dit zou op 2 manieren kunnen bijvoorbeeld met de parse args waar we de gebruiker kunnen limiteren in zijn keuze. Of door een if statement die bij kleine grids een andere regeling gebruikt.

De kleuren van de cellen.

- 0 (Lege cell) - Donker Paars
- 1 (Gezonde cell) - Roze
- 2 (Tumor cell) - Donker paars rood

**Gompertz uitleg**
Het gompertz model wat we nu hanteren is niet voldoende ondersteund door actuele wetenschappelijke literatuur. Het "Klassieke gompertz model" word alleen vindbaar via wikipedia behandeld en door de kans voor editing op deze website is het goed om dit in twijfel te nemen. Hierom gaan we met de volgende formule verder:


$$
\frac{dV}{dt} = c * V * ln(\frac{V_{max}}{V})
$$

$c =$ the specific growth rate per t

$V_{max}$ = maximum achievable volume of the tumor, known as the carrying capacity.

$ln(\frac{V_{max}}{V})$ = a natural logarithmic braking factor which drives the growth rate to zero when V apporaches $V_{max}$

Deze formule is deels uit de volgende papers (Zanotti & Faria, 2025), (Murphy et al., 2016)

We willen ook in de simulatie kunnen zien welke dag het is dus hier gaan we nu mee bezig.

In de simulatie met de basis waarden, groeien de cellen heel snel en de tumor bijna niet wat goed lijkt voor nu, maar we moeten kijken of dit de bedoeling is voor elke tumor aangezien niet elke cel regeneratief is zoals lever cellen dat zijn.


## Reflectie

Er zijn een paar dingen die verbeterd kunnen worden aan de simulatie.

### Code optimalisatie

Wanneer we het hebben over optimale code gebruiken we de grote-O notatie, en als we naar de verschillende loops kijken. Laten we de volgende files behandelen: environment.py, analysis.py & gui.py

**environment.py**

In de functie "create grid" zitten 2 nested for loops met if statements wat onze de complexiteit O(rows x cols) geeft, de tijd complexiteit is niet echt te verbeteren we kunnen wel de code aanpassen om in plaats van 3 for loops 2 te gebruiken met een is else statement.

In de functie "get neighbours" is O(1) omdat het altijd exact 4 elementen zoekt; links, recht, boven en onder. Dus ongeacht hoe groot het grid is zal hij alleen deze 4 elementen zoeken. Deze functie zou dus niet sneller kunnen.

In de functie "do single time step" zit de loop door de grid de eerder genoemde O(rows x cols) binnen de loop wordt get neighbours aangeroepen de O(1) en dat valt mee echter wordt de tumor history lijst aan het eind weer aangeroepen wat ons nogmaals O(rows x cols). Dus we doen in deze functie O(rows x cols) + O(rows x cols), we runnen in deze eene functie dus 2 maal door het gehele grid. Bij elke tijdstap.

Dit is te fiksen door nadat het nieuwe grid gemaakt is (na de initiele loop dus) door allecellen heen te lopen met een for loop en een count bij te houden.

```python
for row in new_grid:
        for cel in row:
            if cel == 2:
                tumor_count += 1

TUMOR_HISTORY.append(tumor_count)
```

We stroomlijnen hier dus de list comprehension aan het einde van de functie.

**analysis.py**

De functies zijn in het algemeen niet heel zwaar, write_pdf is O(1) omdat je een loop hebt die door 3 figuren loopt elke keer.

Plot_tumor_history, gompertz_plot, gompertz & overlay_plot, lezen/genereren data van lengte R of S die op een plot getekent worden wat de complexiteit O(R) of O(S) linear dus.

Comparing plots is de groete boosdoener, door zijn onnodige dubbel werk het bestand lezen in het geval de tumor_history.csv (dit lezen is erg traag) dit kan gefixt worden door de data 1 keer in te lezen en in een variabele of dataframe op te slaan. Nog een boos doener is de dubbele berekening van gompertz waar gompertz_plot en gompertz hetzelfde berekenen

De algemene fix is om dit sneller te maken is dus alle berekeningen 1 maal doen, en alle csv's en andere data 1 maal inlezen.


**gui.py**

De gameloop is best standaard, dus de gameloop is niet veel aan te veranderen voor optimalisatie. Hetzelfde voor de gui_window dit hangt het meeste af van hoe groot de grid. De gui window is dus O(rows * cols) en de gameloop is O(steps * rows * cols).

In de loop wordt de surface constant opnieuw aangeroepen dus dit buiten de while loop te zetten zal de efficientie verbeteren.


# Model validatie en bug fixes

De `gompertz()` functie had een beetje een probleem ik zocht een netjese makkelijk implementeerbare versie te vinden hier van. Dit was de analytische formule.

$$N(t)=N_{max}⋅e^{−b⋅e ^{−c⋅t}}$$

Deze formule had echter niet genoeg wetenschappelijke backing in recente papers. Waardoor naar een nieuwe formule gezocht werd.

**Bug 1: formule-mismatch.** In een eerdere versie implementeerde `gompertz()`:

$$\frac{dV}{dt} = a \cdot V \cdot \ln\!\left(\frac{N_{max}}{V+c}\right)$$

Met een extra `+c`-term binnen de logaritme. Deze term kwam niet voor in de formule die geciteerde had uit (Zanotti & Faria, 2025; Murphy et al., 2016). De `+c` verschuift het evenwicht van het model: `dV/dt = 0` wordt bereikt bij `V = N_max - c` in plaats van `V = N_max`. De code crashte niet en de curve zag er nog steeds uit als een S-curve, wat het makkelijk maakte om de fout over het hoofd te zien.

Na de formule en paper nog maals te lezen zag ik dat de N_max - c niet in de paper zat er dus dat ondanks er nog een S curve te zien is dit niet betekent dat de formule correct is.


**Bug 2: ongebruikte parameter.** Na het verwijderen van de `+c`-term bleven `a` en `c` los naast elkaar in de functie-signature staan, terwijl alleen `a` nog gebruikt werd in de berekening.

Ik heb de functie opgeschoond tot één parameter `c` (de groeisnelheid), consistent met de notatie $c$ uit mijn literatuur, en deze parameter overal in de codebase (`gompertz_plot`, `overlay_plot`, `comparing_plots`, `gui.py`, `main.py`, `setup.py`) doorgevoerd.

**Bug 3: orde-grootte-fout in een default-waarde.** De default voor `param-c` stond op `1.00494` in plaats van `0.00494` — een typefout bij het overnemen van de eerder berekende kans (0,00494) voor de HCC-verdubbelingstijd. Het gevolg was: met `c=1.00494` bereikte de gesimuleerde tumor het maximum (`N_max`) al binnen 5-10 dagen, in plaats van de verwachte ~140 dagen. De Gompertz rate constante is dus zeer gevoelig.








## Kwantitatieve validatie

Om te checken of mijn model daadwerkelijk de literatuurwaarde benadert, reken ik de werkelijke verdubbelingstijd uit de gesimuleerde Gompertz-curve terug, en vergelijk die met de 130-150 dagen uit (Nathani et al., 2020; Nathani et al., 2021; Wigg et al., 2010).


In [1]:

from analysis import gompertz

N_max = 2500  # 50 x 50 grid
c = 0.00494   # huidige default in setup.py

values = gompertz(max_steps=400, N_max=N_max, c=c)
V0 = values[0]
doubling_day = next(i for i, v in enumerate(values) if v >= 2 * V0)
print(f"Gemodelleerde verdubbelingstijd: {doubling_day} dagen")
print(f"Literatuurwaarde (Nathani et al.): 130-150 dagen")


Gemodelleerde verdubbelingstijd: 20 dagen
Literatuurwaarde (Nathani et al.): 130-150 dagen


**Resultaat:** met de huidige standaard parameter c: 0,00494geeft het model een verdubbelingstijd van ongveer **20 dagen** — 7 keer sneller dan de 130-150 dagen uit de literatuur.

Dit verschil ontstaat omdat ik de dagelijkse ABM-deelkans (`tumor_growth_chance = 0,00494` kans per tijdstap) heb hergebruikt als de Gompertz-rate-constante `c` (een *continue* groeisnelheid in een differentiaalvergelijking).

Deze twee grootheden zijn niet uitwisselbaar, ook al zijn ze afgeleid uit dezelfde verdubbelingstijd-berekening. O

M de Gompertz-curve daadwerkelijk op 140 dagen te krijgen, moeten we dus `0,00494` verkleinen met 7 wat dus `c ≈ 0,00067` geeft in plaats van `0,00494`.

Dit is een belangrijke beperking van mijn huidige model: de koppeling tussen de ABM-deelkans en de ODE-parameter is niet een-op-een, en een toekomstige versie zou `c` apart moeten calibreren (bijvoorbeeld met `scipy.optimize.curve_fit` op de eigen ABM-simulatiedata) in plaats van de letterlijke kans te hergebruiken.



## Model aannamens en beperkingen

- **Oneven biopten** Er wordt aangenomen dat hetgeen wat je ziet op de simulatie begind als een biopt wat afgesneden is waarin elke pixel 1 cell is. De ruis is gespiegeld wat niet realistisch is.
- **1 tumor cell** Elke simulatie wordt gestart met 1 tumor cell wat dus 1 pixel is. we gaan er dus van uit dat elke soort kanker die de gebruiker wil of gaat simuleren begint met 1 tumor cell. Kanker cellen die anders werken zullen niet te simuleren zijn en de gebruiker weet ook de cell proliferatie aan te passen aan de hand van de typecellen.
- **N_max.** als totale grid grootte, N_max. wordt gelijkgesteld aan het totaal aantal cellen wat op de grid zich bevindt omdat we rows * cols doen om dit te berekenen. Dit zegt dat een biopt volledig met kan transformeren naar tumorweefsel in realiteit zijn er limitaties aan de grootte van een tumor dit komt door vasculatuur, omliggend weefsel en immuumrespons. Hiermee wordt niet rekening gehouden in deze simulatie en de tumor kan hier on eindig door groeien.


### Reflectie op het modelleerproces

1 van de belangerijkste leerpunten uit dit project is dat een model dat werkt (niet crashed) en een goed ogend figuur produceerd (een nette S-curve) nog niet betekent dat het ook de juiste vergelijking implementeert of dat de parameters de juiste orde van grootte hebben.

De typefout in `param-c` (`1,00494` i.p.v. `0,00494`) liet bijvoorbeeld een tumor binnen een week volgroeien zonder enige foutmelding — de simulatie "werkte" gewoon, maar modelleerde iets totaal anders dan bedoeld.

Een tweede leerpunt is dat het hergebruiken van één literatuurgetal (de verdubbelingstijd) voor zowel een discrete ABM-kans als een continue ODE rateconstante een verleidelijke maar onjuiste afkorting is

Zoals de kwantitatieve validatie liet zien, levert dit een factor 7 verschil op in het resultaat.


Voor toekomstig modelleeronderzoek in oncologie zou ik:
1. elke ODE-implementatie expliciet term-voor-term terugtoetsen aan de brontekst voordat ik de curve visueel beoordeel.
2. parameters die uit verschillende wiskundige formalismes komen (discreet vs. continu) apart calibreren, bijvoorbeeld via curve-fitting op simulatiedata, in plaats van literatuurgetallen direct te kopiëren tussen modeltypes
3. vroeger in het proces een kwantitatieve validatiestap inbouwen, in plaats van pas achteraf te checken of de uitkomst "redelijk" oogt.




# References
_Common cancer types_. (2025, May 6). Cancer.gov. https://www.cancer.gov/types/common-cancers

Yin, A., Moes, D. J. A., Van Hasselt, J. G., Swen, J. J., & Guchelaar, H. (2019). A review of mathematical models for tumor dynamics and treatment resistance evolution of solid tumors. _CPT Pharmacometrics & Systems Pharmacology_, _8_(10), 720–737. https://doi.org/10.1002/psp4.12450

Srinivasa, K. G., Siddesh, G. M., & Manisekhar, S. R. (Eds.). (2020). _Statistical modelling and machine learning principles for bioinformatics: Techniques, tools, and applications_. Springer Singapore. https://doi.org/10.1007/978-981-15-8640-8

Jain, R. K., Martin, J. D., & Stylianopoulos, T. (2014). The Role of Mechanical Forces in Tumor Growth and Therapy. Annual Review Of Biomedical Engineering, 16(1), 321–346. https://doi.org/10.1146/annurev-bioeng-071813-105259

Lung Cancer Foundation of America. (2024, 13 januari). Types of Lung Cancer | Lung Cancer Foundation of America. LCFA. https://lcfamerica.org/about-lung-cancer/diagnosis/types/

Jiang, B., Han, D., van der Aalst, C. M., Lancaster, H. L., Vonder, M., Gratama, J.-W. C., Silva, M., Field, J. K., de Koning, H. J., Heuvelmans, M. A., & Oudkerk, M. (2024). Lung cancer volume doubling time by computed tomography: A systematic review and meta-analysis. European Journal of Cancer, 212, Article 114339. https://doi.org/10.1016/j.ejca.2024.114339

Kalra, A., Yetiskul, E., Wehrle, C. J., & Tuma, F. (2023, 1 mei). Physiology, liver. StatPearls - NCBI Bookshelf. https://www.ncbi.nlm.nih.gov/books/NBK535438/

Levy, G., Bomze, D., Heinz, S., Ramachandran, S. D., Noerenberg, A., Cohen, M., Shibolet, O., Sklan, E., Braspenning, J., & Nahmias, Y. (2015). Long-term culture and expansion of primary human hepatocytes. Nature Biotechnology, 33(12), 1264–1271. https://doi.org/10.1038/nbt.3377

Wigg, A. J., Palumbo, K., & Wigg, D. R. (2010). Radiotherapy for hepatocellular carcinoma: Systematic review of radiobiology and modeling projections indicate reconsideration of its use. Journal Of Gastroenterology And Hepatology, 25(4), 664–671. https://doi.org/10.1111/j.1440-1746.2009.06126.x

Nathani, P., Gopal, P., Rich, N., Yopp, A., Yokoo, T., John, B., Marrero, J., Parikh, N., & Singal, A. G. (2020). Hepatocellular carcinoma tumour volume doubling time: a systematic review and meta-analysis. Gut, 70(2), gutjnl-2020. https://doi.org/10.1136/gutjnl-2020-321040

Nathani, P., Gopal, P., Rich, N. E., Yopp, A., Yokoo, T., John, B., Marrero, J. A., Parikh, N. D., & Singal, A. G. (2021). Hepatocellular carcinoma tumor volume doubling time: A systematic review and meta-analysis. Gut, 70(2), 401–407. https://doi.org/10.1136/gutjnl-2020-321040

Zanotti, Y. P., & Faria, H. A. M. (2025, 22 augustus). CLASSICAL MATHEMATICAL MODELS OF POPULATION GROWTH FOR PREDICTING CELL CULTURE IN BIOREACTORS. https://periodicos.newsciencepubl.com/editoraimpacto/article/view/7496

Murphy, H., Jaafari, H., & Dobrovolny, H. M. (2016). Differences in predictions of ODE models of tumor growth: a cautionary example. BMC Cancer, 16(1), 163. https://doi.org/10.1186/s12885-016-2164-x